In [ ]:
!pip install "swig"
!pip install "stable-baselines3==2.0.0a5"
!pip install "gymnasium[box2d]"
!pip install 'shimmy>=0.2.1'


In [ ]:
!pip3 install pyvirtualdisplay

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

In [ ]:
# Create the environment
np.random.seed(0)
env = gym.make("LunarLander-v2")

In [ ]:
# env = make_vec_env('LunarLander-v2', n_envs=16)

Policy 1 : Proximal Policy Optimizatio (PPO)

In [ ]:
# We use MultiLayerPerceptron (MLPPolicy) because the input is a vector,
# We added some parameters to accelerate the training
model = PPO(
    policy = 'MlpPolicy',
    env = env,
    n_steps = 1024,
    batch_size = 64,
    n_epochs = 4,
    gamma = 0.999,
    gae_lambda = 0.98,
    ent_coef = 0.01,
    verbose=1)

Generate an observation

In [ ]:
# Then we reset this environment
observation, info = env.reset()

for _ in range(20):
    # Take a random action
    action = env.action_space.sample()
    print("Action taken:", action)

    # Do this action in the environment and get
    # next_state, reward, terminated, truncated and info
    observation, reward, terminated, truncated, info = env.step(action)
    print(info)

    # If the game is terminated (in our case we land, crashed) or truncated (timeout)
    if terminated or truncated:
        # Reset the environment
        print("Environment is reset")
        observation, info = env.reset()

env.close()

Training the model

In [ ]:
# Train it for 1,000,000 timesteps
model.learn(total_timesteps=1000000)
# Save the model
model_name = "ppo-LunarLander-v2"
model.save(model_name)

  Custom Logic : Recording Training Data (Observation States/ Features)

In [ ]:
import csv
import stable_baselines3 as sb3
from stable_baselines3.common.callbacks import BaseCallback
import pandas as pd
class CustomTrainingCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(CustomTrainingCallback, self).__init__(verbose)
        self.observations = []
        self.rewards = []

    def _on_step(self) -> bool:
        self.observations.append(self.locals['new_obs'])  # Access the observation
        self.rewards.append(self.locals['rewards'])  # Access the reward

        return True

Training

In [ ]:
import numpy as np

# Create the custom callback
callback = CustomTrainingCallback()
# Train the model with the custom callback
model.learn(total_timesteps=10000, callback=callback)
# Save the model
model_name = "ppo-LunarLander-v2"
model.save(model_name)
# Convert observations and rewards to DataFrames
obs_array = np.array(callback.observations)
obs_array = obs_array.reshape(obs_array.shape[0], -1)  # Reshape to 2D
obs_df = pd.DataFrame(obs_array)
# obs_df = pd.DataFrame(np.array(callback.observations))
reward_df = pd.DataFrame({'reward': callback.rewards})

# Save observations and rewards to CSV files
obs_df.to_csv('training_data.csv', index=False)
reward_df.to_csv('training_rewards.csv', index=False)



Evaluating the trained Agent

In [ ]:
eval_env = gym.make("LunarLander-v2")
# Evaluate the model with 10 evaluation episodes and deterministic=True
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

In [ ]:
reward_df.head()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the rewards data
reward_df = pd.read_csv('training_rewards.csv')

# Convert the reward column from string representation of lists to floats
reward_df['reward'] = reward_df['reward'].apply(lambda x: float(x.strip('[]')))

# Calculate the rolling mean reward
reward_df['rolling_mean_reward'] = reward_df['reward'].rolling(window=10).mean()

# Mean reward and standard deviation from evaluation
# mean_reward = -616.47
# std_reward = 402.14

# Plot the results
plt.figure(figsize=(14, 7))
plt.plot(reward_df.index, reward_df['reward'], label='Step Reward', alpha=0.3, color='blue')
plt.plot(reward_df.index, reward_df['rolling_mean_reward'], label='Rolling Mean Reward (100 steps)', color='red')
plt.axhline(y=200, color='green', linestyle='--', label='Solved Threshold (200)')
plt.axhline(y=mean_reward, color='orange', linestyle='--', label=f'Mean Eval Reward ({mean_reward})')
plt.fill_between(reward_df.index, mean_reward - std_reward, mean_reward + std_reward, color='orange', alpha=0.2, label=f'Std Dev ({std_reward})')
plt.xlabel('Steps')
plt.ylabel('Score')
plt.title('Step Rewards and Rolling Mean Rewards over Time')
plt.legend()
plt.grid(True)

# Save the plot
plt.savefig('step_rewards_plot_with_mean_eval.png')
plt.show()


In [ ]:
# Get the last rolling mean reward
# Calculate the rolling mean of rewards over the last 100 steps
rolling_mean_rewards = reward_df['reward'].rolling(window=100).mean()

last_rolling_mean = rolling_mean_rewards.iloc[-1]
# Print the results
print(f"The last rolling mean reward (over 100 steps) is: {last_rolling_mean}")

    Render

In [ ]:
env = gym.make("LunarLander-v2", render_mode="rgb_array")
observation = env.reset()
print(observation)
obs, other = observation
print(obs)

In [ ]:
import pandas as pd
import numpy as np

observation_df = pd.read_csv('training_data.csv')
columns_to_iterate = observation_df.columns
for index, obs in observation_df[columns_to_iterate].iloc[:100, :].iterrows():
    obs_array = obs.to_numpy()
    print(obs_array)

In [ ]:
import os
import gym
import matplotlib.pyplot as plt
os.environ["IMAGEIO_FFMPEG_EXE"] = r"."
import imageio
from IPython.display import clear_output

# Create a list to store frames
frames = []

for index, obs in observation_df[columns_to_iterate].iloc[:100000, :].iterrows():
    obs_array = obs.to_numpy()
    action, _states = model.predict(obs_array)
    observations, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        # Reset the environment
        print("Environment is reset")
        observation, info = env.reset()

    clear_output(wait=True)
    frame = env.render()
    frames.append(frame)

env.close()

# Save the frames as a GIF
imageio.mimsave('lunar_lander_without_imposter.gif', frames)


Entropy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
entropies = []
from scipy.stats import entropy
# Iterate over the columns of your DataFrame
for feature in obs_df.columns:
    # Calculate the PDF of the Gaussian distribution for each column
    entropy = stats.entropy(np.histogram(obs_df[feature], bins=8)[0])
    print(f"Entropy for '{feature}': {entropy:.4f}")
    entropies.append(entropy)

In [ ]:
combined_single_entropy = sum(entropies)
print(f'Combined Single Entropy (Sum): {combined_single_entropy:.4f}')

In [ ]:
# Plot the line graph
plt.plot(obs_df.columns, entropies, marker='o')
plt.xlabel('Features')
plt.ylabel('Entropy')
plt.title('Entropy of all 8 Features')
plt.xticks(rotation=45)
plt.show()

Joint Entropy

In [ ]:
import itertools
joint_entropies = []
features = obs_df.columns

# Generate all possible combinations of 2 features
combinations = list(itertools.combinations(features, 2))
print(len(combinations))
print(combinations)

In [ ]:
joint_entropies = []
joint_entropies_list = []
for feature1, feature2 in combinations:
    # Calculate the joint histogram
    hist, x_edges, y_edges = np.histogram2d(obs_df[feature1], obs_df[feature2], bins=[10, 10])
    # Calculate the joint probability distribution
    p = hist / np.sum(hist)
    joint_entropy = stats.entropy(p.flatten(), base=2)
    print(f"Joint Entropy for '{feature1}' and '{feature2}': {joint_entropy:.4f}")
    joint_entropies.append(joint_entropy)
    joint_entropies_list.append((feature1, feature2, joint_entropy))

In [ ]:
combined_joint_entropy = sum(joint_entropies)
print(combined_joint_entropy)

In [ ]:
# Plot the line graph
feature_pairs = [f"{feature1} - {feature2}" for feature1, feature2, _ in joint_entropies_list]
plt.plot(feature_pairs, joint_entropies, marker='o')
plt.xlabel('Feature Combinations')
plt.ylabel('Joint Entropy')
plt.title('Joint Entropy without y-coord')
plt.xticks(rotation=45)
plt.show()

Heatmap

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming joint_entropies is a list of tuples where each tuple is (feature1, feature2, joint_entropy)
# You can convert it to a pivot table for easier visualization
import pandas as pd
data = pd.DataFrame(joint_entropies_list, columns=['Feature1', 'Feature2', 'Joint Entropy'])
pivot_table = data.pivot(index='Feature1', columns='Feature2', values='Joint Entropy')

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(pivot_table, annot=True, fmt=".4f", cmap='viridis')
plt.title('Joint Entropy of Feature Combinations with all 8')
plt.show()


In [ ]:
# The trained agent
trained_env = gym.make("LunarLander-v2", render_mode="rgb_array")
obs = trained_env.reset()
states8, other = obs
trained_obs = []
for i in range(1000):
    action, _states = model.predict(states8)
    obs, rewards, terminated, truncated, info = env.step(action)
    trained_obs.append(obs)
    env.render()
print('Trained observations: ',trained_obs)

In [ ]:
import pandas as pd
import numpy as np
df_trained_obs = pd.DataFrame(trained_obs)
df_trained_obs.head()
df_trained_obs.to_csv('trained_observations_all.csv')

In [ ]:
observation_df = pd.read_csv('trained_observations_all.csv')
for index, obs in observation_df.iterrows():
    obs_array = obs.to_numpy()
    print(obs_array)

In [ ]:
import os
import gym
import matplotlib.pyplot as plt
os.environ["IMAGEIO_FFMPEG_EXE"] = r"MyFFMPEG_PATH"
from IPython.display import clear_output

# env = gym.make("LunarLander-v2", render_mode="rgb_array")
# env.action_space.seed(42)
# observation = env.reset()
# print(observation)
# obs, other = observation
# print(obs)

for index, obs in observation_df.iterrows():
    obs_array = obs.to_numpy()
#     print(obs_array)
    action, _states = model.predict(obs_array)
    print('action', action)
    observations, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        # Reset the environment
        print("Environment is reset")
        observation, info = env.reset()
        
#     clear_output(wait=True)
#     plt.imshow( env.render() )
#     plt.show()

env.close()

In [ ]:
distance_to_target = 0.0
speed,time = 0.0, 0.0
df_target = pd.DataFrame()
target_var = []

for i, s in df_trained_obs.iterrows():
    distance = np.sqrt(s[0]**2 + s[1]**2)
    speed = np.sqrt(s[2]**2 + s[3]**2)
    time = distance/speed

    distance_and_time = {'Distance': distance, 'Time' : time }
    target_var.append(distance_and_time)

target_data = pd.DataFrame(target_var)
df_target = pd.concat([df_target, target_data], axis=1)

In [ ]:
df_target.describe()

In [ ]:
print(df_target.to_string())


SHAP

In [ ]:
%pip install shap

In [ ]:
import shap
import pandas as pd

In [ ]:
X = df.drop(columns=['target_variable_column_name'])
y = df['target_variable_column_name']